# RLM vs a plain LLM call: build an Excel report from 140 MB of IMF data

The deliverable is a formatted Excel workbook: a pivot of the 10 highest-inflation
countries with yearly averages, a merged title cell, bold headers, and a median
row. The source is the IMF's CPI dataflow pulled live from their public SDMX API:
about 140 MB of CSV, 1.5 million observation rows, 194 countries.

A plain LLM call cannot produce a file at all, and it cannot see the data. So the
baseline gets a generous version of the task: the same question, answered as JSON,
with as much raw CSV in its prompt as reasonably fits. The RLM arms get the same
model plus an interpreter, and must build the real workbook. A deterministic
DuckDB query grades everything at the end.

In [ ]:
%pip install -q fabric-rlm[analytics] openpyxl

## Download the data

One GET against the IMF SDMX 3.0 API. No API key. About 140 MB; the IMF server
usually delivers it in under a minute. The cell skips the download if the file is
already there, and prefers Lakehouse `Files` when one is mounted.

In [ ]:
import os, urllib.request

DATA_DIR = "/lakehouse/default/Files" if os.path.isdir("/lakehouse/default/Files") else "."
DATA_PATH = os.path.join(DATA_DIR, "imf_cpi.csv")
URL = (
    "https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/5.0.0/*.*.*.*.*"
    "?c%5BTIME_PERIOD%5D=ge:2017-01-01+le:2026-12-31"
)

if not os.path.exists(DATA_PATH):
    req = urllib.request.Request(URL, headers={"Accept": "application/vnd.sdmx.data+csv"})
    with urllib.request.urlopen(req, timeout=900) as r, open(DATA_PATH, "wb") as fh:
        while chunk := r.read(1 << 20):
            fh.write(chunk)
print(f"{os.path.getsize(DATA_PATH) / 1e6:.0f} MB at {DATA_PATH}")

## Pick the model

In a Fabric notebook, `FabricLM` uses the capacity's built-in Azure OpenAI
endpoint: no key and nothing to provision. Outside Fabric, the same cell falls
back to OpenRouter using an `OPENROUTER_API_KEY` environment variable.

In [ ]:
def make_lm(model):
    """Fabric first: the capacity's built-in Azure OpenAI endpoint, no key.
    Outside Fabric, fall back to OpenRouter (set OPENROUTER_API_KEY)."""
    from fabric_rlm import FabricLM
    try:
        return FabricLM(model)
    except Exception:
        import dspy
        return dspy.LM(
            f"openrouter/openai/{model}",
            api_key=os.environ["OPENROUTER_API_KEY"],
            api_base="https://openrouter.ai/api/v1",
            max_tokens=16000,
            temperature=1.0,
        )

lm_big = make_lm("gpt-5.1")

## The task

In [ ]:
REPORT_PATH = os.path.join(DATA_DIR, "cpi_report.xlsx")

TASK = """You are given IMF CPI data in SDMX-CSV format (one observation per row)
at data_file. Build an Excel report at report_path (create the file with openpyxl).

Data selection: rows with INDEX_TYPE='CPI', COICOP_1999='_T',
TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT', FREQUENCY='M'. TIME_PERIOD looks like
'2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

The workbook has a single sheet named 'Report':
- A1:G1 merged, containing exactly: Average year-over-year CPI inflation (%), 2021-2025
- Row 2 headers, bold: Country, 2021, 2022, 2023, 2024, 2025, Avg 2021-2025
- Rows 3 to 12: the 10 qualifying countries with the highest five-year average of
  their monthly YoY values, sorted descending by that average. Each year column is
  that calendar year's average of the 12 monthly values; 'Avg 2021-2025' is the
  average of all 60 monthly values. Write numbers rounded to 2 decimals.
- Row 13: column A = 'Median (all qualifying countries)', column G = the median
  across ALL qualifying countries of their five-year averages, 2 decimals.

Save to report_path, then SUBMIT with n_countries (the count of qualifying
countries) and median_avg (the row-13 median value)."""

## Arm 1: plain LLM call

200,000 characters of the file (under 0.15 percent) plus the question. When we
ran this, gpt-5.1 burned about 109,000 prompt tokens and then answered honestly
that the task is impossible from the slice: the visible rows do not even include
the required series.

In [ ]:
import time

PLAIN_PROMPT = """The text below is the beginning of a large CSV of IMF CPI data in
SDMX-CSV format (one observation per row). The full file is about 140 MB; only
this slice fits in your context.

Rows with INDEX_TYPE='CPI', COICOP_1999='_T', TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT',
FREQUENCY='M' are monthly year-over-year all-items CPI inflation. TIME_PERIOD looks
like '2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

Report, as JSON only:
{"n_countries": <count of qualifying countries>,
 "top10": [[country, avg2021, avg2022, avg2023, avg2024, avg2025, avg_5yr], ...],
 "median_avg": <median across all qualifying countries of their 5-year averages>}
top10 = the 10 qualifying countries with the highest 5-year average of monthly YoY
values, sorted descending, yearly values = that year's average of 12 monthly values,
all numbers to 2 decimals."""

head = open(DATA_PATH, encoding="utf-8").read(200_000)
t0 = time.time()
plain_text = lm_big(f"{PLAIN_PROMPT}\n\n--- FILE SLICE ---\n{head}")[0]
plain_seconds = time.time() - t0
plain_usage = (lm_big.history[-1].get("usage") or {}) if getattr(lm_big, "history", None) else {}
print(plain_text[:1500])

## Arm 2: the same model, through the RLM

The model writes DuckDB and openpyxl code in the subprocess: aggregate 1.5
million rows, pivot by year, merge the title cell, bold the headers, save the
workbook. When we ran this, gpt-5.1 finished in 2 turns and about 19,000 total
tokens: less than a fifth of what the failed plain call spent.

In [ ]:
from fabric_rlm import File, RLM

t0 = time.time()
rlm = RLM.task(
    task=TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": REPORT_PATH},
    outputs=["n_countries", "median_avg"],
    lm=lm_big,
    skills=["data_exploration", "excel_modify"],
    max_turns=10,
    timeout=600.0,
)
result_big = rlm.run()
rlm_big_seconds = time.time() - t0
result_big.payload

## Arm 3: a mini model, through the RLM

The point of giving the model an interpreter is that the model no longer has to
be huge. When we ran this, gpt-5-mini (about 5x cheaper than gpt-5.1) produced
the identical correct workbook in 4 turns.

In [ ]:
REPORT_PATH_MINI = os.path.join(DATA_DIR, "cpi_report_mini.xlsx")
lm_mini = make_lm("gpt-5-mini")

t0 = time.time()
rlm_mini = RLM.task(
    task=TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": REPORT_PATH_MINI},
    outputs=["n_countries", "median_avg"],
    lm=lm_mini,
    skills=["data_exploration", "excel_modify"],
    max_turns=10,
    timeout=600.0,
)
result_mini = rlm_mini.run()
rlm_mini_seconds = time.time() - t0
result_mini.payload

## Ground truth and scoreboard

A deterministic DuckDB pivot computes the true report. Each workbook is then
reloaded with openpyxl and checked structurally (sheet name, merged `A1:G1`,
exact title, bold headers) and numerically (all 60 pivot values and the median
within 0.02). The plain arm is graded generously: it only has to name the ten
correct countries anywhere in its answer.

In [ ]:
import duckdb, statistics

con = duckdb.connect()
rows = con.execute(f"""
WITH obs AS (
    SELECT COUNTRY, substr(TIME_PERIOD, 1, 4) AS yr, OBS_VALUE
    FROM read_csv_auto('{DATA_PATH}')
    WHERE INDEX_TYPE = 'CPI' AND COICOP_1999 = '_T'
      AND TYPE_OF_TRANSFORMATION = 'YOY_PCH_PA_PT' AND FREQUENCY = 'M'
      AND substr(TIME_PERIOD, 1, 4) BETWEEN '2021' AND '2025'
      AND OBS_VALUE IS NOT NULL
), complete AS (
    SELECT COUNTRY FROM obs GROUP BY COUNTRY HAVING count(*) = 60
), yearly AS (
    SELECT o.COUNTRY, yr, avg(OBS_VALUE) AS y_avg
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY, yr
), fivey AS (
    SELECT o.COUNTRY, avg(OBS_VALUE) AS avg5
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY
)
SELECT f.COUNTRY,
  max(CASE WHEN yr = '2021' THEN y_avg END), max(CASE WHEN yr = '2022' THEN y_avg END),
  max(CASE WHEN yr = '2023' THEN y_avg END), max(CASE WHEN yr = '2024' THEN y_avg END),
  max(CASE WHEN yr = '2025' THEN y_avg END), max(avg5)
FROM fivey f JOIN yearly y ON f.COUNTRY = y.COUNTRY
GROUP BY f.COUNTRY ORDER BY max(avg5) DESC
""").fetchall()

truth = {
    "n_countries": len(rows),
    "top10": [[r[0]] + [round(v, 2) for v in r[1:]] for r in rows[:10]],
    "median_avg": round(statistics.median(r[6] for r in rows), 2),
}
truth["top10"]

In [ ]:
def grade_workbook(path):
    from openpyxl import load_workbook
    if not os.path.exists(path):
        return {"file_created": False}
    checks = {"file_created": True}
    wb = load_workbook(path)
    checks["sheet_named_Report"] = "Report" in wb.sheetnames
    ws = wb["Report"] if "Report" in wb.sheetnames else wb.active
    checks["A1_G1_merged"] = "A1:G1" in [str(r) for r in ws.merged_cells.ranges]
    checks["title_text"] = (ws["A1"].value or "").strip() == "Average year-over-year CPI inflation (%), 2021-2025"
    hdr = [str(ws.cell(row=2, column=c).value) for c in range(1, 8)]
    checks["headers"] = hdr == ["Country", "2021", "2022", "2023", "2024", "2025", "Avg 2021-2025"]
    checks["headers_bold"] = all(ws.cell(row=2, column=c).font.bold for c in range(1, 8))
    ok = True
    for i, want in enumerate(truth["top10"]):
        r = 3 + i
        if str(ws.cell(row=r, column=1).value).strip() != want[0]:
            ok = False
            continue
        for j in range(6):
            try:
                if abs(float(ws.cell(row=r, column=2 + j).value) - want[1 + j]) > 0.02:
                    ok = False
            except (TypeError, ValueError):
                ok = False
    checks["top10_values"] = ok
    med = ws.cell(row=13, column=7).value
    try:
        checks["median_row"] = abs(float(med) - truth["median_avg"]) <= 0.02
    except (TypeError, ValueError):
        checks["median_row"] = False
    return checks


def summarize(name, checks, tokens, seconds):
    passed = all(checks.values())
    detail = "" if passed else " (" + ", ".join(k for k, v in checks.items() if not v) + ")"
    print(f"{name:<22} {str(passed):<9} {tokens:>10,} {seconds:>9.1f}{detail}")


truth_top10 = [r[0] for r in truth["top10"]]
plain_checks = {"names_all_correct": all(c in plain_text for c in truth_top10), "file_created": False}
plain_tokens = (plain_usage.get("prompt_tokens") or 0) + (plain_usage.get("completion_tokens") or 0)

big_checks = grade_workbook(REPORT_PATH)
big_tokens = (result_big.total_prompt_tokens or 0) + (result_big.total_completion_tokens or 0)

mini_checks = grade_workbook(REPORT_PATH_MINI)
mini_tokens = (result_mini.total_prompt_tokens or 0) + (result_mini.total_completion_tokens or 0)

print(f"{'arm':<22} {'passed':<9} {'tokens':>10} {'seconds':>9}")
summarize("plain call (gpt-5.1)", plain_checks, plain_tokens, plain_seconds)
summarize("RLM (gpt-5.1)", big_checks, big_tokens, rlm_big_seconds)
summarize("RLM (gpt-5-mini)", mini_checks, mini_tokens, rlm_mini_seconds)

## What just happened

The plain call could not name a single correct country: the slice it saw covers
a fraction of one country's rows, and no amount of model quality fixes missing
data. Both RLM arms computed the pivot from all 1.5 million rows and wrote a
correctly formatted workbook, verified cell by cell against ground truth. In our
run the scoreboard read:

| arm | passed | tokens | seconds |
|---|---|---|---|
| plain call (gpt-5.1) | False | 109,480 | 8.4 |
| RLM (gpt-5.1) | True | 18,937 | 33.5 |
| RLM (gpt-5-mini) | True | 44,732 | 66.7 |

Both RLM arms spent fewer tokens than the failed plain call, because the raw
bytes never enter the model's context: only its code, the code's output, and
its summaries do. And the mini model's pass is the deeper point: with an
interpreter attached, a small cheap model outperforms a flagship model without
one. The README section "When to use an RLM (and when not to)" covers when this
trade is worth it.